# 글자 단위 BiLSTM 시퀀스 라벨링 — 단어 분할 (Colab)

**연습 기법** (IOAI 2025 GAITE **Combinatorial Word Segmentation** 와 동일): 붙어 있는 글자열을 **각 글자마다
0/1 을 다는 시퀀스 라벨링**으로 분할한다. 원문제는 독일어 복합어('Fußballspieler')의 각 글자에 **1=단어 끝**,
**0=시작/중간** 을 예측한다. 권장 기법은 **임베딩(Embedding) + (Bi)LSTM** — 글자 임베딩 → LSTM → 글자별 이진분류.
지표는 **단어별 F1 평균**.

**대회**: [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) (재난 트윗). 트윗 텍스트의 **공백을
제거**해 글자열을 만들고, 각 글자에 **1=단어의 마지막 글자(=경계)** 를 라벨링한다 → 독일어 복합어 분할과 *동일한
과제*(붙은 글자열을 단어로 가르기). 07(TF-IDF 분류)·12(BERT 분류)·16(문장임베딩)과 달리, 여기선 **글자 단위
토큰 분류**(per-character tagging)를 한다.

**핵심 흐름 & 배울 점**:
1. **데이터 구성** — 공백 제거한 글자열 + 글자별 경계 라벨(1=단어 끝).
2. **글자 임베딩 + BiLSTM** — 양방향 문맥으로 각 글자의 경계 여부 예측.
3. **패딩 마스킹** — 가변 길이 시퀀스를 배치로 묶고 손실/지표에서 패딩 제외.
4. **단어별 F1**(원문제 지표) 평가 + 예측 경계로 글자열을 **단어로 복원**해 확인.

> ⚙️ GPU 권장(작은 BiLSTM, T4 ~1분). 데이터가 작아 CPU 도 가능.
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 💡 이 과제(글자열→단어 분할)는 트윗 *분류*(nlp-getting-started 리더보드)와 다른 **구성된 연습**이라 캐글 CSV 제출
> 대신, 원문제처럼 **분할 예측 JSON** 과 단어별 F1 로 평가합니다.

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 트윗 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_file("nlp-getting-started", "train.csv", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 구성 — 공백 제거 글자열 + 경계 라벨
각 트윗의 알파벳 단어들을 소문자로 이어 붙여 **글자열**을 만들고, 각 글자에 **1=단어의 마지막 글자(경계)**,
나머지 **0** 을 라벨링한다. (원문제의 `1=단어 끝` 과 동일 규칙)

In [ ]:
import re, numpy as np, pandas as pd
df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
MAXLEN = 80

def make_example(text):
    words = re.findall(r"[a-zA-Z]+", str(text).lower())
    chars, labels = [], []
    for w in words:
        for k, ch in enumerate(w):
            chars.append(ch); labels.append(1 if k == len(w) - 1 else 0)   # 1 = 단어 끝
    return chars[:MAXLEN], labels[:MAXLEN]

seqs = [make_example(t) for t in df["text"]]
seqs = [(c, l) for c, l in seqs if 4 <= len(c) <= MAXLEN and sum(l) >= 2]   # 경계 2개 이상

vocab = {"<pad>": 0}
for c, _ in seqs:
    for ch in c: vocab.setdefault(ch, len(vocab))
enc = lambda c: [vocab[ch] for ch in c]

ex_c, ex_l = seqs[0]
print("예시 글자열:", "".join(ex_c))
print("경계 라벨   :", "".join(str(x) for x in ex_l))
print("examples:", len(seqs), "| vocab:", len(vocab))

## 3. 배치 구성 (패딩 + 마스크) & train/valid 분할
가변 길이라 배치마다 최대 길이에 맞춰 0 패딩하고, **마스크**로 패딩 위치를 손실·지표에서 제외한다.

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.RandomState(0); idx = rng.permutation(len(seqs)); cut = int(len(seqs) * 0.9)
train_data = [seqs[i] for i in idx[:cut]]; val_data = [seqs[i] for i in idx[cut:]]

def batchify(data, bs=64, shuffle=False):
    order = np.random.permutation(len(data)) if shuffle else np.arange(len(data))
    for i in range(0, len(order), bs):
        chunk = [data[j] for j in order[i:i+bs]]; m = max(len(c) for c, _ in chunk)
        X = np.zeros((len(chunk), m), "int64"); Y = np.zeros((len(chunk), m), "float32"); M = np.zeros((len(chunk), m), "float32")
        for j, (c, l) in enumerate(chunk):
            X[j, :len(c)] = enc(c); Y[j, :len(c)] = l; M[j, :len(c)] = 1
        yield torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M)
print("train:", len(train_data), "| valid:", len(val_data))

## 4. 모델 — 글자 임베딩 + 양방향 LSTM (원문제의 Embedding+LSTM)
각 글자를 임베딩하고 **BiLSTM** 으로 양방향 문맥을 본 뒤, 글자마다 경계 로짓 1개를 출력한다.

In [ ]:
import torch.nn as nn

class CharTagger(nn.Module):
    def __init__(self, vocab_size, emb=32, hid=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hid * 2, 1)
    def forward(self, x):
        return self.fc(self.lstm(self.emb(x))[0]).squeeze(-1)   # (batch, time)

print("CharTagger 파라미터:", sum(p.numel() for p in CharTagger(len(vocab)).parameters()))

## 5. 학습 & 평가 (마스킹 BCE, 단어별 F1)

In [ ]:
from sklearn.metrics import f1_score
torch.manual_seed(0)
model = CharTagger(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
bce = nn.BCEWithLogitsLoss(reduction="none")
EPOCHS = 8

for epoch in range(1, EPOCHS + 1):
    model.train()
    for X, Y, M in batchify(train_data, shuffle=True):
        X, Y, M = X.to(device), Y.to(device), M.to(device)
        opt.zero_grad()
        loss = (bce(model(X), Y) * M).sum() / M.sum()           # 패딩 제외 평균
        loss.backward(); opt.step()
    # 검증: 단어별 F1 평균
    model.eval(); f1s = []
    with torch.no_grad():
        for X, Y, M in batchify(val_data, bs=128):
            p = (torch.sigmoid(model(X.to(device))) > 0.5).cpu().numpy()
            Yn, Mn = Y.numpy(), M.numpy()
            for j in range(len(X)):
                n = int(Mn[j].sum())
                f1s.append(f1_score(Yn[j, :n].astype(int), p[j, :n].astype(int), zero_division=1))
    print(f"epoch {epoch} | val 단어별 F1 평균 = {np.mean(f1s):.4f}")
print("\n→ 글자 임베딩 + BiLSTM 으로 글자별 경계를 라벨링 — 원문제(독일어 복합어 분할)와 동형.")

## 6. 예측 경계로 글자열을 단어로 복원 (시각 확인)

In [ ]:
def segment(chars):
    model.eval()
    with torch.no_grad():
        x = torch.tensor([enc(chars)], dtype=torch.long, device=device)
        pred = (torch.sigmoid(model(x))[0] > 0.5).cpu().numpy()
    out = ""
    for ch, b in zip(chars, pred):
        out += ch + (" " if b else "")
    return out.strip()

for c, _ in val_data[:8]:
    print(f"{''.join(c):40s} → {segment(c)}")

## 7. 분할 예측 JSON 저장 (원문제 `submissionval.json` 형식)
원문제는 `{글자열: [0/1 배열]}` 형식의 분할 예측을 제출한다. 검증셋 예측을 같은 형식으로 저장한다.

In [ ]:
import json
model.eval(); pred_dict = {}
with torch.no_grad():
    for c, _ in val_data:
        x = torch.tensor([enc(c)], dtype=torch.long, device=device)
        pred = (torch.sigmoid(model(x))[0] > 0.5).int().cpu().tolist()
        pred_dict["".join(c)] = pred
out_path = os.path.join(WORK_DIR, "submissionval.json")
json.dump(pred_dict, open(out_path, "w"))
print("Saved:", out_path, "| 예측 글자열 수:", len(pred_dict))

In [ ]:
try:
    from google.colab import files
    files.download(out_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submissionval.json 위치:", out_path)

## 🏁 정리
**기법 요약** (IOAI 2025 Word Segmentation): 붙은 글자열을 **글자 단위 시퀀스 라벨링**으로 분할 — **글자 임베딩 +
BiLSTM** 으로 각 글자에 `1=단어 끝` 예측, **패딩 마스킹**으로 가변 길이 처리, **단어별 F1** 평가. 원문제(독일어
복합어)와 동형이며, 여기선 트윗 텍스트의 공백 복원으로 연습했다. 더 끌어올리려면: LSTM 층/은닉차원↑, CRF 헤드
(이웃 라벨 제약), subword/문자 n-gram 특징, 더 많은 에폭·학습률 스케줄. (이 구성된 과제는 nlp-getting-started
리더보드(트윗 분류)와 무관하므로 캐글 CSV 제출은 없음 — 원문제처럼 분할 JSON·F1 로 평가.)